# Python Agent Framework - Metacognition Example

This notebook demonstrates how to build an AI travel agent with metacognitive capabilities using the Python Agent Framework. The agent maintains awareness of user preferences across the conversation and uses this context to provide personalized recommendations without repeatedly asking for the same information.

## Install Required Packages

In [ ]:
%pip install agent-framework-azure-ai -U

## Import Required Packages

In [2]:
print("hello world")

hello world


In [3]:
import json
import os

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

## Define Tool Functions

Tool functions provide the agent with capabilities to retrieve destination information and flight times.

In [4]:
def get_destinations() -> str:
    """Provides a list of vacation destinations.
    
    Returns:
        A formatted string listing available vacation destinations
    """
    return """
    Barcelona, Spain
    Paris, France
    Berlin, Germany
    Tokyo, Japan
    New York, USA
    """

def get_flight_times(destination: str) -> str:
    """Provides available flight times for a destination.
    
    Args:
        destination: The destination to check flight times for
        
    Returns:
        Flight times for the specified destination
    """
    flight_times = {
        "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
        "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
        "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
        "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
        "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
    }

    # Extract just the city name from input that might contain country
    city = destination.split(',')[0].strip()

    if city in flight_times:
        times = ", ".join(flight_times[city])
        return f"Flight times for {city}: {times}"
    else:
        return f"No flight information available for {city}."

## Load Environment Variables and Configure Azure OpenAI Client

In [5]:
load_dotenv()

# Load environment variables
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")

print("Environment Variables:")
print("=" * 50)
print(f"AZURE_AI_FOUNDRY_ENDPOINT: {azure_endpoint}")
print(f"AZURE_OPENAI_API_VERSION: {api_version}")
print(f"AZURE_AI_FOUNDRY_MODEL: {model_name}")
print(f"AZURE_AI_FOUNDRY_API_KEY: {'*' * 20 if api_key else None}")
print("=" * 50)

Environment Variables:
AZURE_AI_FOUNDRY_ENDPOINT: https://ibecfoundry.openai.azure.com/
AZURE_OPENAI_API_VERSION: 2024-02-01
AZURE_AI_FOUNDRY_MODEL: gpt-4o
AZURE_AI_FOUNDRY_API_KEY: ********************


In [6]:
# Configure the Azure AI Agent Client
if not api_key:
    raise ValueError(
        "AZURE_AI_FOUNDRY_API_KEY environment variable is required. "
        "Please set it in your .env file or environment."
    )

# Azure OpenAI Chat Client using Python Agent Framework
chat_client = AzureOpenAIChatClient(
    endpoint=azure_endpoint,
    api_key=api_key,
    deployment_name=model_name
)

print("✅ Azure OpenAI Chat Client configured successfully!")

✅ Azure OpenAI Chat Client configured successfully!


## Create the Agent with Metacognitive Instructions

The agent is configured with detailed instructions that enable it to remember user preferences and apply them across the conversation without repeatedly asking for the same information.

In [7]:
AGENT_NAME = "TravelAgent"
AGENT_INSTRUCTIONS = """
You are Flight Booking Agent that provides information about available flights and gives travel activity suggestions when asked.
Travel activity suggestions should be specific to customer, location and amount of time at location.

You have access to the following tools to help users plan their trips:
1. get_destinations: Returns a list of available vacation destinations that users can choose from.
2. get_flight_times: Provides available flight times for specific destinations.

Your process for assisting users:
- When users first inquire about flight booking with no prior history, ask for their preferred flight time ONCE.
- MAINTAIN a customer_preferences object throughout the conversation to track preferred flight times.
- When a user books a flight to any destination, RECORD their chosen flight time in the customer_preferences object.
- For ALL subsequent flight inquiries to ANY destination, AUTOMATICALLY apply their existing preferred flight time without asking.
- NEVER ask about time preferences again after they've been established for any destination.
- When suggesting flights for a new destination, explicitly say: "Based on your previous preference for [time] flights, I recommend..."
- Only after showing options matching their preferred time, ask if they want to see alternative times.
- After each booking, UPDATE the customer_preferences object with any new information.
- ALWAYS mention which specific preference you used when making a suggestion.

Guidelines:
- Use the exact destination names when using tools (Barcelona, Paris, Berlin, Tokyo, New York)
- Respond in a helpful and enthusiastic manner about travel possibilities
- Always seek feedback to ensure your suggestions meet the user's expectations
- Acknowledge when a request falls outside your capabilities
- For better formatting, always display flight times in a list format
- When giving any timed suggestions, reflect if the time frames are reasonable. Respond again if not.

Your goal is to help users explore vacation options efficiently and make informed travel decisions by understanding their preferences and providing tailored recommendations.
"""

# Create the agent with tools
agent = chat_client.create_agent(
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    tools=[get_destinations, get_flight_times],
    tool_choice="auto"
)

print(f"✅ Created agent: {AGENT_NAME}")

✅ Created agent: TravelAgent


## Run the Agent with Streaming

This demonstration shows how the agent remembers user preferences across multiple interactions and applies them automatically to subsequent requests.

In [8]:
user_inputs = [
    "Book me a flight to Barcelona",
    "I prefer a later flight",
    "That is too late, choose the earliest flight",
    "I want to leave the same day, give me some suggestions of things to do in Barcelona during my layover if I take the last flight out",
    "I am stressed this wont be enough time"
]

async def main():
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response = []
        function_calls = []

        print(f"\n{'='*80}")
        print(f"User: {user_input}")
        print(f"{'='*80}\n")

        # Run the agent with streaming
        async for chunk in agent.run_stream(user_input):
            # Collect text responses
            if chunk.text:
                full_response.append(chunk.text)
                print(chunk.text, end="", flush=True)
            
            # Collect function call information if available
            if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
                for tool_call in chunk.tool_calls:
                    function_calls.append(f"Calling function: {tool_call.function.name}({tool_call.function.arguments})")
            
            # Collect function results if available
            if hasattr(chunk, 'tool_outputs') and chunk.tool_outputs:
                for tool_output in chunk.tool_outputs:
                    function_calls.append(f"\nFunction Result:\n\n{tool_output}")

        print("\n")

        # Build HTML output for Jupyter display
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{AGENT_NAME}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()


User: Book me a flight to Barcelona

Can you letCan you let me know your preferred me know your preferred flight time (e flight time (e.g.g., morning., morning, afternoon, evening, afternoon, evening, or night, or night)?)? It'll help me suggest It'll help me suggest the best options for you! Once the best options for you! Once I know I know this, I can start this, I can start building your travel preferences for future building your travel preferences for future flights. flights.






User: I prefer a later flight

Thank you for sharing your preferenceThank you for sharing your preference for later flights! Let for later flights! Let me record your preference in me record your preference in the system.

Would you like to explore the system.

Would you like to explore available vacation destinations available vacation destinations, or do you already have, or do you already have a specific location in mind a specific location in mind? Let me help? Let me help you plan you plan your trip! your trip!






User: That is too late, choose the earliest flight

It seems that you've referenced aIt seems that you've referenced a flight without prior context in flight without prior context in this conversation! Could you this conversation! Could you clarify the clarify the destination you're planning to fly destination you're planning to fly to? to? I’ll I’ll then find the earliest flight then find the earliest flight options for you.

 options for you.

Would you also like assistanceWould you also like assistance discovering available vacation destinations discovering available vacation destinations?

?




User: I want to leave the same day, give me some suggestions of things to do in Barcelona during my layover if I take the last flight out

The last flight to Barcelona isThe last flight to Barcelona is at 10:45 at 10:45 PM. If PM. If you you choose to take this flight choose to take this flight, here, here are some exciting layover activities are some exciting layover activities you can do you can do in Barcelona in Barcelona during during the day:

### Suggestions:
 the day:

### Suggestions:
1. **Explore La Ramb1. **Explore La Rambla**  
   Stroll through this iconic streetla**  
   Stroll through this iconic street filled with vibrant street filled with vibrant street performers, market stalls, performers, market stalls, shops shops, and eateries.

, and eateries.

2. **2. **Visit Sagrada Família**  
Visit Sagrada Família**  
     A must A must-see-see architectural marvel by architectural marvel by Antoni Antoni Gaudí Gaudí – perfect for – perfect for some some quick sightseeing a


User: I am stressed this wont be enough time

I'm here to helpI'm here to help! Could you! Could you let me know let me know a bit more a bit more about about what you're what you're planning planning? For? For example:

1. example:

1. Are you looking to Are you looking to book a flight book a flight?
2?
2.. Where do you want Where do you want to go ( to go (or explore)?
or explore)?
3. How much time do you3. How much time do you have for your trip?

Don't worry—I'll assist you with have for your trip?

Don't worry—I'll assist you with getting everything figured out getting everything figured out! 😊! 😊





## Continue the Conversation - Testing Metacognition

Now let's test if the agent remembers the user's preferences from the previous conversation when booking a flight to a different destination.

In [ ]:
async def continue_chat():
    # Continue the conversation with new user input
    user_input = "Book me a flight to Paris"
    
    html_output = (
        f"<div style='margin-bottom:10px'>"
        f"<div style='font-weight:bold'>User:</div>"
        f"<div style='margin-left:20px'>{user_input}</div></div>"
    )

    full_response = []
    function_calls = []

    print(f"\n{'='*80}")
    print(f"User: {user_input}")
    print(f"{'='*80}\n")

    # Run the agent with streaming
    async for chunk in agent.run_stream(user_input):
        # Collect text responses
        if chunk.text:
            full_response.append(chunk.text)
            print(chunk.text, end="", flush=True)
        
        # Collect function call information if available
        if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
            for tool_call in chunk.tool_calls:
                function_calls.append(f"Calling function: {tool_call.function.name}({tool_call.function.arguments})")
        
        # Collect function results if available
        if hasattr(chunk, 'tool_outputs') and chunk.tool_outputs:
            for tool_output in chunk.tool_outputs:
                function_calls.append(f"\nFunction Result:\n\n{tool_output}")

    print("\n")

    # Build HTML output for Jupyter display
    if function_calls:
        html_output += (
            "<div style='margin-bottom:10px'>"
            "<details>"
            "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
            "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
            "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
            f"{chr(10).join(function_calls)}"
            "</div></details></div>"
        )

    html_output += (
        "<div style='margin-bottom:20px'>"
        f"<div style='font-weight:bold'>{AGENT_NAME}:</div>"
        f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
    )

    display(HTML(html_output))

await continue_chat()


User: Book me a flight to Paris

Could you please letCould you please let me know what me know what time you'd prefer to time you'd prefer to fly? For fly? For example, morning example, morning, afternoon, or, afternoon, or evening evening? This will? This will help help me match the best options me match the best options for you!

 for you!



: 